<a href="https://colab.research.google.com/github/hmdaalln/image-denoising-capstone/blob/main/Image_Denoising.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Denoising

## Project Overview :
In this notebook, we prepare the SIDD dataset before training the image denoising model.
The goal is to understand the dataset structure and visualize noisy and clean images.

## Problem Statement

Image noise can reduce image quality and negatively affect many computer vision applications.

Our objective is to build a model that learns how to transform noisy images into clean images while keeping edges and textures as natural as possible.

## Project Objectives

The main objectives of this project are:

- Explore and understand the SIDD dataset.
- Prepare the dataset for training.
- Train a deep learning model for image denoising.
- Evaluate the model using PSNR and SSIM.
- Compare the model performance and visualize the results.

## Dataset Description

This project uses the Smartphone Image Denoising Dataset (SIDD).

The dataset contains pairs of noisy and clean images captured using different smartphones under various lighting conditions. It is specifically designed for image denoising tasks.

## Workflow
1. Import Libraries
2. Explore the Dataset
3. Data Preprocessing
4. Create DataLoaders

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 1. Import Libraries

In [1]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import copy
import time
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

## 2. Dataset Exploration

In this section, we explore the SIDD dataset to understand its structure and examine a sample pair of noisy and clean images before training the model.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Extract the dataset
zip_path = "/content/drive/MyDrive/cap/SIDD_Small_sRGB_Only.zip"
extract_path = "/content/SIDD"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
# Path to the extracted dataset
dataset_path = "/content/SIDD/SIDD_Small_sRGB_Only/Data"

# Read all folders
image_folders = os.listdir(dataset_path)

print("Number of samples:", len(image_folders))
print("First sample folder:", image_folders[0])
print("Second  sample folder:", image_folders[1])

In [ ]:
# Open one sample folder
sample_image = os.path.join(dataset_path, image_folders[2])

# Show the files inside the scene
files = os.listdir(sample_image)

print("Files in the first sample:")
print(files)

In [ ]:
# Read the images
noisy_image = Image.open(os.path.join(sample_image, "NOISY_SRGB_010.PNG"))
clean_image = Image.open(os.path.join(sample_image, "GT_SRGB_010.PNG"))

# Display the images
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(noisy_image)
plt.title("Noisy Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(clean_image)
plt.title("Clean Image")
plt.axis("off")

plt.show()

### Observation

The SIDD dataset contains pairs of noisy and clean images. Each noisy image has a corresponding clean image that will be used as the target during model training.

## 3. Data Preprocessing
In this section, we prepare the SIDD image pairs for model training.  
First, we split the sample folders into training, validation, and testing sets.  
The images will later be converted into smaller patches and normalized.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the folders into training and remaining data
train_folders, temp_folders = train_test_split(
    image_folders,
    test_size=0.30,
    random_state=42
)

# Split the remaining folders into validation and testing
val_folders, test_folders = train_test_split(
    temp_folders,
    test_size=0.50,
    random_state=42
)

# Check the number of samples in each set
print("Training samples:", len(train_folders))
print("Validation samples:", len(val_folders))
print("Testing samples:", len(test_folders))

### Prepare the Image Pairs

In this step, we create a dataset class to load each noisy image with its matching clean image. We also crop the same area from both images and convert them into tensors.

In [ ]:
class SIDDDataset(Dataset):

    def __init__(
        self,
        dataset_path,
        folders,
        patch_size=128
        patches_per_image=8,
        augment=False

        ):

        # Save dataset information
        self.dataset_path = dataset_path
        self.folders = folders
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.augment = augment

    def __len__(self):

      # Calculate the total number of patches
      # Each image folder can generate several random patches
        return len(self.folders) * self.patches_per_image

    def __getitem__(self, index):

      # Identify which image folder to use
      folder_index = index // self.patches_per_image

        # Open one sample folder
        sample_path = os.path.join(
            self.dataset_path,
            self.folders[index]
        )

        # Find the noisy and clean image files
        files = os.listdir(sample_path)

        noisy_file = [f for f in files if "NOISY" in f][0]
        clean_file = [f for f in files if "GT" in f][0]

        # Open the images
        noisy_image = Image.open(
            os.path.join(sample_path, noisy_file)
        ).convert("RGB")

        clean_image = Image.open(
            os.path.join(sample_path, clean_file)
        ).convert("RGB")

        # Crop the same area from both images
        i, j, h, w = transforms.RandomCrop.get_params(
            noisy_image,
            output_size=(self.patch_size, self.patch_size)
        )

        noisy_image = TF.crop(noisy_image, i, j, h, w)
        clean_image = TF.crop(clean_image, i, j, h, w)

        # Apply paired data augmentation
        # The same transformation must be applied to both images
        if self.augment:

           # Random horizontal flip
            if torch.rand(1).item() > 0.5:
                noisy_image = TF.hflip(noisy_image)
                clean_image = TF.hflip(clean_image)

            # Random vertical flip
            if torch.rand(1).item() > 0.5:
                noisy_image = TF.vflip(noisy_image)
                clean_image = TF.vflip(clean_image)

        # Convert images to tensors
        noisy_image = TF.to_tensor(noisy_image)
        clean_image = TF.to_tensor(clean_image)

        # Calculate the noise residual
        noise_target = noisy_image - clean_image

        return {
            "noisy": noisy_image,
            "clean": clean_image,
            "noise_target": noise_target
        }

In [ ]:
# Create the dataset
train_dataset = SIDDDataset(
    dataset_path,
    train_folders,
    patch_size=128
    patches_per_image=8,
    augment=True
)

# Get one sample
noisy_patch, clean_patch = train_dataset[0]

# Check the output
print("Training samples:", len(train_dataset))
print("Noisy shape:", noisy_patch.shape)
print("Clean shape:", clean_patch.shape)

## 4. Create DataLoaders
In this step, we create DataLoaders for the training, validation, and testing datasets. The DataLoader loads the images in batches during training.

In [ ]:
# Create the datasets
train_dataset = SIDDDataset(dataset_path, train_folders)
val_dataset = SIDDDataset(dataset_path, val_folders)
test_dataset = SIDDDataset(dataset_path, test_folders)

In [ ]:
# Create the dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

In [ ]:
# Check one batch
noisy_batch, clean_batch = next(iter(train_loader))

print("Noisy batch:", noisy_batch.shape)
print("Clean batch:", clean_batch.shape)
print("Noise target:", batch["noise_target"].shape)

## 5. Select the device



In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = DnCNN().to(device)

print(model)

**Why this code?**
- This code checks whether a GPU is available.

- If a GPU is available, the model and data will be moved to the GPU to make training faster. Otherwise, the code will use the CPU.

**What do we noticed?**

If the output is:

**Device: cuda**

the notebook is using a GPU.

If the output is:

**Device: cpu**

the model will still run, but training will be slower.

## 6.1. Build the DnCNN Model

In this section, we build the DnCNN model from scratch.

DnCNN uses convolutional layers to learn the noise residual in an image. Instead of directly predicting the clean image, the model predicts the noise that should be removed. The denoised image is obtained by subtracting the predicted noise from the noisy input.

In [ ]:
class DnCNN(nn.Module):

    def __init__(
        self,
        image_channels=3,
        depth=17,
        features=64
    ):
        super(DnCNN, self).__init__()

        layers = []

        # First convolution layer
        layers.append(
            nn.Conv2d(
                in_channels=image_channels,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=True
            )
        )

        layers.append(
            nn.ReLU(inplace=True)
        )

        # Middle convolution layers
        for _ in range(depth - 2):

          layers.append(
            nn.Conv2d(
                in_channels=image_channels,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False
            )
        )

            layers.append(
                nn.BatchNorm2d(features)
            )

            layers.append(
                nn.ReLU(inplace=True)
            )

        # Final convolution layer
        layers.append(
            nn.Conv2d(
                in_channels=image_channels,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False
            )
        )

        self.model = nn.Sequential(*layers)

    def forward(self, x):

        predicted_noise = self.model(x)

        return predicted_noise

**Why this code?**

**DnCNN** uses residual learning.

Instead of predicting the clean image directly, the model predicts the noise contained in the image.

The true noise is calculated as:

- **Noise target = Noisy image − Clean image**

The final denoised image is then calculated as:

- **Denoised image = Noisy image − Predicted noise**

This approach allows the model to focus on learning the unwanted noise.

**What we noticed?**



## 6.2. Create the Model


In [ ]:
model = DnCNN(
    image_channels=3,
    depth=17,
    features=64
).to(device)

print(model)

**Why this code?**

This cell creates the **DnCNN** model and moves it to the selected device.

The values mean:

- image_channels=3: the input images are RGB.
- depth=17: the network contains 17 convolutional layers.
- features=64: the hidden layers use 64 feature maps.

**What do we noticed?**


## 6.3. Count Model Parameters

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

**Why this code?**

This code calculates the number of model parameters.

The parameter count helps us understand the model size and complexity. It will also be useful later when comparing DnCNN with the pretrained model.


**What do we noticed?**

## 6.4. Test the Forward Pass

In [ ]:
batch = next(iter(train_loader))

noisy_batch = batch["noisy"].to(device)

with torch.no_grad():

    predicted_noise = model(noisy_batch)

print("Input shape:", noisy_batch.shape)
print("Predicted noise shape:", predicted_noise.shape)

**Why this code?**

This test confirms that the model can process a batch before training begins.

Using torch.no_grad() prevents gradient calculations because this is only a test.


**What do we noticed?**

## Training Setup and Training Loop

## 6.5. Define Training Configuration

In [ ]:
CONFIG = {
    "learning_rate": 0.001,
    "epochs": 30,
    "early_stopping_patience": 5
}

**Why this code?**

This dictionary stores the main training settings in one place.

- learning_rate: controls the size of weight updates.
- epochs: maximum number of complete training cycles.
- early_stopping_patience: number of epochs to wait before stopping when validation loss does not improve.

**What do we noticed?**

## 6.6. Create Output Folders

In [ ]:
CHECKPOINT_ROOT = ""
METADATA_ROOT = ""

os.makedirs(
    CHECKPOINT_ROOT,
    exist_ok=True
)

os.makedirs(
    METADATA_ROOT,
    exist_ok=True
)

print("Folders created.")


**Why this code?**

These folders are used to save:

- The best model checkpoint,
- Training history,
- Evaluation results.

The files are saved in Google Drive so they are not lost when the Colab session ends.

**What do we noticed?**

## 6.7. Define Loss Function

In [ ]:
criterion = nn.MSELoss()

**Why this code?**

The model predicts the noise residual.

MSE loss measures the average squared difference between:

- The predicted noise,
- The true noise residual.

A smaller MSE value means that the predicted noise is closer to the actual noise

**What do we noticed?**

## 6.8. Define Optimizer

In [ ]:
optimizer = Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"]
)

**Why this code?**

The optimizer updates the model parameters during training.

Adam is used because it is a common and stable optimizer for deep learning models.

**What do we noticed?**

## 6.9. Define Learning Rate Scheduler

In [ ]:
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

**Why this code?**

The scheduler monitors validation loss.

**What do we noticed?**

## 6.10. Define one Epoch Function

In [ ]:
def run_epoch(
    model,
    loader,
    training
):

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_samples = 0

    for batch in tqdm(
        loader,
        leave=False
    ):

        noisy = batch["noisy"].to(device)
        target_noise = batch[
            "noise_target"
        ].to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):

            predicted_noise = model(noisy)

            loss = criterion(
                predicted_noise,
                target_noise
            )

            if training:

                loss.backward()

                optimizer.step()

        batch_size = noisy.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

    average_loss = (
        total_loss
        / total_samples
    )

    return average_loss

**Why this code?**

This function runs one complete pass through either the training or validation dataset.

It is used for both stages to avoid repeating similar code.

**Step explanation**

- model.train() enables training behavior.

- model.eval() changes the model to evaluation mode.

- optimizer.zero_grad() removes gradients from the previous batch.

- predicted_noise = model(noisy) performs the forward pass.

- loss.backward() calculates gradients.

- optimizer.step() updates the model weights.


**What do we noticed?**

## 6.11. Train the Model & Save the Best Checkpoint

In [ ]:
history = []

best_val_loss = float("inf")
patience_counter = 0

best_model_path = os.path.join(
    CHECKPOINT_ROOT,
    "dncnn_best.pth"
)

for epoch in range(
    1,
    CONFIG["epochs"] + 1
):

    start_time = time.perf_counter()

    train_loss = run_epoch(
        model,
        train_loader,
        training=True
    )

    val_loss = run_epoch(
        model,
        val_loader,
        training=False
    )

    scheduler.step(val_loss)

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.perf_counter()
        - start_time
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr,
        "time_seconds": epoch_time
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.3f} | "
        f"Validation Loss: {val_loss:.3f} | "
        f"LR: {current_lr:.2e} | "
        f"Time: {epoch_time:.1f}s"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_loss": val_loss,
                "config": CONFIG
            },
            best_model_path
        )

        print("Best model saved.")

    else:

        patience_counter += 1

    if (
        patience_counter
        >= CONFIG[
            "early_stopping_patience"
        ]
    ):

        print("Early stopping triggered.")

        break


**Why this code?**

This loop trains the model for several epochs.

After every epoch, it:

- Calculates training loss,
- Calculates validation loss,
- Updates the learning rate,
- Stores the results,
- Saves the best model,
- Checks whether early stopping is required.

**Why save the best model?**

The last epoch is not always the best one.

The model with the lowest validation loss is saved because it is expected to generalize better.

**Why early stopping?**

Early stopping prevents unnecessary training when validation performance no longer improves.

It may also reduce overfitting and save GPU time.

**What do we noticed?**

## 6.12. Save Training History

In [ ]:
history_df = pd.DataFrame(history)

history_path = os.path.join(
    METADATA_ROOT,
    "dncnn_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

history_df.tail()

**Why this code?**

The training results are stored in a table and saved as a CSV file.

The file can later be used in:

- The report,
- The presentation,
- Model comparison,
- GitHub documentation.

**What do we noticed?**

## Evaluation and Visualization

## 6.13. Plot the Loss Curves